# AE-TIRT Basic Tutorial

This notebook provides a concise, standards-aligned walkthrough of three core workflows:

1. **How to simulate forced-choice data**;
2. **How to train and evaluate an AE-TIRT model**;
3. **How to analyze real data (template with a runnable example pattern)**.

The API style is consistent with:
- `examples/01_basic_usage.py`
- `examples/02_simulation_study.py`
- `examples/03_real_data_analysis.py`
- `examples/04_batch_experiments.py`
- `examples/05_paper_real_data_example.py`

---

## Tutorial Scope

This tutorial focuses on a simple, practical pipeline for first-time users:
- one synthetic dataset,
- one model training run,
- one real-data analysis template.

For large-scale simulation studies and multi-condition experiments, see `examples/02_simulation_study.py` and `examples/04_batch_experiments.py`.

## 1) Environment Setup

Run the following cell to import required libraries and define a compute device.

> Tip: If you use a GPU-enabled PyTorch installation, training will run on CUDA automatically.

In [12]:
import random
import numpy as np
import torch

from ae_tirt import AE_TIRT, Sim_data_TIRT, evaluate_model, train_model


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}, seed: {SEED}")

Using device: cpu, seed: 42


## 2) Simulate Data

We generate synthetic forced-choice responses with known latent traits. This is the recommended starting point for validating your training setup before moving to real data.

In [13]:
sim = Sim_data_TIRT(
    npersons=500,
    ntraits=5,
    nblocks_per_trait=12,
    nitems_per_block=3,
    weight_sign=0.5,
    w_range=(0.65, 0.95),
    b_range=(-1.0, 1.0),
    comb_blocks="random",
).simulate()

print("Synthetic data generated.")
print(f"responses shape: {sim.responses.shape}")
print(f"theta shape: {sim.theta.shape}")
print(f"number of items: {len(sim.item_trait_map)}")
print(f"number of pairs: {len(sim.pair_definitions)}")

Synthetic data generated.
responses shape: (500, 60)
theta shape: (500, 5)
number of items: 60
number of pairs: 60


## 3) Train an AE-TIRT Model

Next, we initialize `AE_TIRT` using the simulated design matrices and train with early stopping.

In [14]:
model = AE_TIRT(
    input_dim=sim.responses.shape[1],
    latent_dim=sim.theta.shape[1],
    item_trait_map=torch.tensor(sim.item_trait_map, dtype=torch.long),
    pair_definitions=torch.tensor(sim.pair_definitions, dtype=torch.long),
    weight_sign=torch.tensor(sim.weight_sign_array, dtype=torch.float32),
    weight_constraint="standardized",
    link_function="probit",
)

history = train_model(
    model=model,
    train_data=sim.responses,
    optimizer_name="adam",
    batch_size=32,
    num_epochs=500,
    learning_rate=1e-3,
    device=device,
    early_stopping_patience=20,
    penalty_weight=sim.responses.shape[1] * 1.0,
)

print(f"Training finished. Epochs run: {len(history['total_loss'])}")

Epoch [10/500], Loss: 1137.3197, Recon Loss: 1130.7362, Z Penalty: 0.1097
Epoch [20/500], Loss: 937.4543, Recon Loss: 907.0048, Z Penalty: 0.5075
Epoch [30/500], Loss: 889.8661, Recon Loss: 851.1675, Z Penalty: 0.6450
Epoch [40/500], Loss: 845.8388, Recon Loss: 797.2025, Z Penalty: 0.8106
Epoch [50/500], Loss: 834.8738, Recon Loss: 785.8723, Z Penalty: 0.8167
Epoch [60/500], Loss: 826.9994, Recon Loss: 778.1072, Z Penalty: 0.8149
Epoch [70/500], Loss: 821.2799, Recon Loss: 772.3469, Z Penalty: 0.8155
Epoch [80/500], Loss: 816.8806, Recon Loss: 768.0208, Z Penalty: 0.8143
Epoch [90/500], Loss: 813.2934, Recon Loss: 764.4335, Z Penalty: 0.8143
Epoch [100/500], Loss: 810.7010, Recon Loss: 761.8016, Z Penalty: 0.8150
Epoch [110/500], Loss: 808.4663, Recon Loss: 759.8056, Z Penalty: 0.8110
Epoch [120/500], Loss: 806.4365, Recon Loss: 758.0211, Z Penalty: 0.8069
Epoch [130/500], Loss: 804.7146, Recon Loss: 756.3541, Z Penalty: 0.8060
Epoch [140/500], Loss: 803.2060, Recon Loss: 755.3792, Z P

## 4) Evaluate on Synthetic Ground Truth

Because synthetic data includes true latent traits, we can compute reconstruction and trait-recovery metrics.

In [15]:
metrics = evaluate_model(model, sim.responses, sim.theta, device=device)

print("Overall trait RMSE:", round(metrics["traits"]["overall"]["rmse"], 4))
print("Overall trait correlation:", round(metrics["traits"]["overall"]["cor"], 4))

Overall trait RMSE: 0.4078
Overall trait correlation: 0.9228


## 5) Data-Input Workflow (Using the Full Simulated Dataset Here)

The same training procedure applies to your own dataset.
In this tutorial cell, we directly reuse the **full simulated dataset** as a stand-in for user-provided data.

Required inputs are still:
- `responses`: shape `(n_persons, n_pairs)`, binary forced-choice outcomes;
- `item_trait_map`: shape `(n_items,)`, trait index per statement;
- `pair_definitions`: shape `(n_pairs, 2)`, statement indices per pair;
- `weight_sign`: shape `(n_items,)`, sign constraints for statement weights.

Without ground-truth traits, skip trait-recovery metrics and report fitted parameter estimates.

In [17]:
# Here we reuse the full simulated dataset as a complete input example.
responses = sim.responses
item_trait_map = sim.item_trait_map
pair_definitions = sim.pair_definitions
weight_sign = sim.weight_sign_array

# Re-seed to keep this section reproducible even when run independently.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Robust latent-dimension inference for 0-based or 1-based trait indexing.
item_trait_min = int(min(item_trait_map))
item_trait_max = int(max(item_trait_map))
latent_dim = item_trait_max if item_trait_min == 1 else item_trait_max + 1

custom_model = AE_TIRT(
    input_dim=responses.shape[1],
    latent_dim=latent_dim,
    item_trait_map=torch.tensor(item_trait_map, dtype=torch.long),
    pair_definitions=torch.tensor(pair_definitions, dtype=torch.long),
    weight_sign=torch.tensor(weight_sign, dtype=torch.float32),
    weight_constraint="standardized",
    link_function="probit",
)

import io
from contextlib import redirect_stdout


with redirect_stdout(io.StringIO()):
    history = train_model(
        model=custom_model,
        train_data=responses,
        optimizer_name="adam",
        batch_size=16,
        num_epochs=500,
        learning_rate=1e-3,
        device=device,
        early_stopping_patience=20,
        penalty_weight=responses.shape[1] * 1.0,
    )

print(f"Training finished. Epochs run: {len(history['total_loss'])}")

# No ground truth is required here.
# Report estimated parameters from the trained model.
params = custom_model.get_trained_parameters()

print("First 5 statement weights:")
for i, (k, v) in enumerate(params["statement_weights"].items()):
    if i >= 5:
        break
    print(f"  item {k}: {v:.4f}")

print("First 5 pair intercepts:")
for i, (k, v) in enumerate(params["pair_intercepts"].items()):
    if i >= 5:
        break
    print(f"  pair {k}: {v:.4f}")

Training finished. Epochs run: 365
First 5 statement weights:
  item 1: 0.9926
  item 2: 0.9909
  item 3: 0.9823
  item 4: 0.9896
  item 5: -0.9881
First 5 pair intercepts:
  pair 1: -0.7851
  pair 2: -0.1348
  pair 3: 0.5788
  pair 4: -0.0727
  pair 5: 0.9322


## 6) Workflow Summary

1. Prepare `responses`, `item_trait_map`, `pair_definitions`, and `weight_sign`.
2. Build `AE_TIRT` with these inputs.
3. Train with `train_model(...)` using the same settings pattern as the simulation section.
4. Report fitted parameters via `get_trained_parameters()`.

For simulation studies and condition sweeps, see:
- `examples/02_simulation_study.py`
- `examples/04_batch_experiments.py`

This completes the minimal AE-TIRT basic tutorial.